In [ ]:
from argparse import Namespace
from tqdm import tqdm
import pandas as pd
import numpy as np
import torch
import glob
import os
import sys
sys.path.append('../resources/library/tropical_cyclone')
from tropical_cyclone.cyclone import BYTETracker

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# define inference directory to draw detections
setup = 'cls_loc_models'
dataset_dir = f'PATH-TO/data/inference/{setup}'
available_models = sorted([folder for folder in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, folder))])
len(available_models)

In [ ]:
# select the model to analyze
selected_model = 'localization_model-classification_model'

# define test years (same as paper)
test_years = [i for i in range(1980,2024)]

# tracker arguments
bbox_size = 25 # 15, 21, 25, 31, 35 # default : 25
track_thresh = 0.7  # Threshold of confidence to consider a detection valid
track_buffer = 1    # Number of frames (6-hourly time-steps) to keep a track marked as "lost"
match_thresh = 0.8  # IoU threshold to associate new tracks to current ones (the higher the farther we look)

# convert lat and lon to row col (considering map as a matrix)
lats, lons = np.linspace(0, 70, 281)[::-1], np.linspace(100, 320, 881)

In [ ]:
# get model directory
model_dir = os.path.join(dataset_dir, selected_model)
# get inference filenames
inference_files = [os.path.join(model_dir, f'{year}.csv') for year in test_years]
model_dir, len(inference_files)

In [ ]:
# load csv files
dataframes = []
for file in inference_files:
    dataframes.append(pd.read_csv(file, index_col=0))
# merge csv files together
detections = pd.concat(dataframes).reset_index(drop=True)
# convert iso time with pandas
detections['ISO_TIME'] = pd.to_datetime(detections['ISO_TIME'])
# add WS as np.inf
detections['WS'] = np.inf
detections.head()

In [ ]:
# convert lat and lon to row col (considering map as a matrix)
lats, lons = np.linspace(0, 70, 281)[::-1], np.linspace(100, 320, 881)
detections['YLAT'] = [np.argwhere(lats==l)[0][0] for l in detections['LAT']]
detections['XLON'] = [np.argwhere(lons==l)[0][0] for l in detections['LON']]
detections['BBOX'] = np.matrix.tolist(np.stack([detections['XLON'] - bbox_size / 2, detections['YLAT'] - bbox_size / 2, detections['XLON'] + bbox_size / 2, detections['YLAT'] + bbox_size / 2], axis=-1))

In [ ]:
tracking_src = f'PATH-TO/data/inference/{setup}/{selected_model}/tracking_TT{track_thresh}-TB{track_buffer}-MT{match_thresh}-BB{bbox_size}.csv'
if not os.path.exists(tracking_src):
    tracker = BYTETracker(track_thresh=track_thresh, track_buffer=track_buffer, match_thresh=match_thresh, frame_rate=1, ratio=1, lats=lats, lons=lons)
    size = (281, 881)
    # for each iso time in the dataset
    for iso_time in tqdm(detections['ISO_TIME'].unique()):
        dets = detections[detections['ISO_TIME']==iso_time]
        output_result = torch.as_tensor(np.c_[np.stack(dets['BBOX']), np.stack(dets['PROB'])])
        tracker.update(output_result, img_info=size, img_size=size, date=iso_time)
    tracks = tracker.create_tracks_dataframe()
    tracks.to_csv(tracking_src)